# 02 - Feature engineering

Builds each fighter's pre-fight history from earlier fights only, then combines red and blue into one row per fight (red minus blue differences).

**Rule:** no feature may use information from the fight it describes, or from later fights.

In [1]:
import os
import numpy as np
import pandas as pd

df = pd.read_csv("../data/raw/stats_processed_all_bouts.csv", sep=";")
df["event_date"] = pd.to_datetime(df["event_date"], format="%d/%m/%Y")
df = df.sort_values("event_date", kind="stable").reset_index(drop=True)
df["fight_id"] = df.index

In [ ]:
## Fight length

mm_ss = df["time"].str.split(":", expand=True).astype(float)
df["fight_seconds"] = (df["round"] - 1) * 300 + mm_ss[0] * 60 + mm_ss[1]

In [ ]:
## One row per fighter per fight

def side_rows(side, other):
    m = df["method"].fillna("")
    won = (df[f"{side}_fighter_result"] == "W").astype(int)
    lost = (df[f"{side}_fighter_result"] == "L").astype(int)
    ko = m.str.contains("KO|TKO", regex=True).astype(int)
    sub = m.str.contains("Submission").astype(int)
    return pd.DataFrame({
        "fight_id": df["fight_id"], "side": side, "event_date": df["event_date"],
        "fighter": df[f"{side}_fighter_name"],
        "won": won, "won_ko": won * ko, "won_sub": won * sub,
        "lost_ko": lost * ko, "lost_sub": lost * sub,
        "minutes": df["fight_seconds"] / 60,
        "kd": df[f"{side}_fighter_KD"].fillna(0),
        "sub_att": df[f"{side}_fighter_sub_att"].fillna(0),
        "ctrl": df[f"{side}_fighter_ctrl"].fillna(0),
    })

long = pd.concat([side_rows("red", "blue"), side_rows("blue", "red")])
long = long.sort_values(["fighter", "event_date", "fight_id"], kind="stable").reset_index(drop=True)
g = long.groupby("fighter", sort=False)

In [ ]:
## History features (current fight excluded)

n = g.cumcount()                                          # number of PRIOR fights
def prior_sum(col):  return g[col].cumsum() - long[col]   # totals minus the current fight
def per_fight(col):  return (prior_sum(col) / n).where(n > 0)

out = pd.DataFrame({"fight_id": long["fight_id"], "side": long["side"], "fighter": long["fighter"]})
out["prior_fights"]      = n
out["win_rate"]          = per_fight("won")
out["ko_win_rate"]       = per_fight("won_ko")
out["sub_win_rate"]      = per_fight("won_sub")
out["ko_loss_rate"]      = per_fight("lost_ko")
out["sub_loss_rate"]     = per_fight("lost_sub")
out["kd_per_fight"]      = per_fight("kd")
out["sub_att_per_fight"] = per_fight("sub_att")
out["ctrl_per_min"]      = (prior_sum("ctrl") / prior_sum("minutes")).where(n > 0)
out["avg_minutes"]       = per_fight("minutes")

# win streak going INTO the fight
losing = (long["won"] == 0).astype(int)
block = losing.groupby(long["fighter"]).cumsum()
streak_after = long.groupby([long["fighter"], block])["won"].cumsum()
out["win_streak"] = streak_after.groupby(long["fighter"]).shift().fillna(0)

# last 3 fights (shifted, so the current fight is excluded)
out["last3_win_rate"] = g["won"].transform(lambda s: s.shift().rolling(3, min_periods=1).mean())

# days since the previous fight
out["days_since_last"] = (long["event_date"] - g["event_date"].shift()).dt.days

In [ ]:
## Age from date of birth

fd = pd.read_csv("../data/raw/raw_fighter_details.csv")
fd["key"] = fd["fighter_name"].str.upper().str.strip()
fd["dob"] = pd.to_datetime(fd["DOB"], format="%b %d, %Y", errors="coerce")
dob = fd.drop_duplicates("key").set_index("key")["dob"]

out["age"] = (long["event_date"] - long["fighter"].str.upper().str.strip().map(dob)).dt.days / 365.25

In [ ]:
## Back to one row per fight

feat_cols = [c for c in out.columns if c not in ("fight_id", "side", "fighter")]
red  = out[out.side == "red"].set_index("fight_id")[feat_cols].add_prefix("red_")
blue = out[out.side == "blue"].set_index("fight_id")[feat_cols].add_prefix("blue_")
fights = df.set_index("fight_id")

pre = pd.DataFrame(index=fights.index)
pre["event_date"] = fights["event_date"]
pre["red_fighter_name"] = fights["red_fighter_name"]
pre["blue_fighter_name"] = fights["blue_fighter_name"]
pre["is_title"] = fights["bout_type"].str.contains("Title", na=False).astype(int)
pre["is_womens"] = fights["bout_type"].str.contains("Women", na=False).astype(int)
pre["scheduled_5_rounds"] = fights["time_format"].str.startswith("5 Rnd").astype(int)
pre["red_stance"] = fights["red_fighter_stance"]
pre["blue_stance"] = fights["blue_fighter_stance"]
pre["height_diff"] = fights["red_fighter_height"] - fights["blue_fighter_height"]
pre["reach_diff"] = fights["red_fighter_reach"] - fights["blue_fighter_reach"]

X = pre.join(red).join(blue)
for c in feat_cols:
    X[f"{c}_diff"] = X[f"red_{c}"] - X[f"blue_{c}"]
X["winner"] = fights["winner"]

In [ ]:
## Keep 2010+ decisive fights and save

model_df = X[X["winner"].isin(["red", "blue"]) & (X["event_date"] >= "2010-01-01")].copy()
model_df["red_won"] = (model_df["winner"] == "red").astype(int)
model_df = model_df.drop(columns="winner")

print(model_df.shape, model_df["red_won"].mean())

os.makedirs("../data/processed", exist_ok=True)
model_df.reset_index(drop=True).to_csv("../data/processed/fights_features.csv", index=False)

(7352, 53) 0.580658324265506


In [ ]:
## Leakage checks

first = n == 0
print("first fights:", first.sum())
print("prior_fights all 0:", (out.loc[first, "prior_fights"] == 0).all())
print("win_rate all NaN:", out.loc[first, "win_rate"].isna().all())
print("win_streak all 0:", (out.loc[first, "win_streak"] == 0).all())

chk = out[out.fighter == "JIM MILLER"].merge(
    long[["fight_id", "side", "event_date", "won"]], on=["fight_id", "side"]
).sort_values("event_date")
chk[["event_date", "won", "prior_fights", "win_rate", "win_streak", "last3_win_rate", "days_since_last", "age"]].head(5)

first fights: 2693
prior_fights all 0: True
win_rate all NaN: True
win_streak all 0: True


,event_date,won,prior_fights,win_rate,win_streak,last3_win_rate,days_since_last,age
0,2008-10-18,1,0,NaN,0.0,NaN,NaN,25.136208
1,2008-12-10,1,1,1.000000,1.0,1.000000,53.0,25.281314
2,2009-03-07,0,2,1.000000,2.0,1.000000,87.0,25.519507
3,2009-07-11,1,3,0.666667,0.0,0.666667,126.0,25.864476
4,2009-09-19,1,4,0.750000,1.0,0.666667,70.0,26.056126
